# SAM3 Instance Segmentation on Fruits360

SAM3 is Meta's unified foundation model for *promptable segmentation* — it builds on SAM 2
by adding **open-vocabulary text prompts**, allowing you to segment any of 270K+ named concepts
alongside the classic visual prompts (points, boxes, masks).

**Key difference from DINOv3 approach:** DINOv3 produces a single cosine-similarity heatmap that
must be manually thresholded and cannot separate touching instances. SAM3 outputs a precise,
per-instance binary mask with a learned IoU confidence score.

**Three prompt modes covered in this notebook:**

| Part | Model | Prompt | When to use |
|---|---|---|---|
| **1 — Text** | `Sam3Model` | Class name string | Open-vocabulary, no manual annotation |
| **2 — Bounding box** | `Sam3TrackerModel` | `[x1, y1, x2, y2]` | Semi-interactive, one object at a time |
| **3 — Point** | `Sam3TrackerModel` | `(x, y)` pixel click | Fastest, least robust |

All heavy functions live in `sam3_seg_utils.py` — this notebook only contains
configuration variables and calls to those functions.

**Prerequisites:** Fruits360 dataset extracted, `apple_pear_1.jpg` scene image,
`transformers >= 4.56.0`, `torch`, `pillow`, `matplotlib`.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import sys; sys.path.insert(0, ".")

In [ ]:
from sam3_seg_utils import (
    MODEL_ID, INSTANCE_COLORS, _color_overlay, _fast_contour,
    load_sam3_model, load_tracker_model,
    segment_by_text, segment_by_bbox, segment_by_point,
    get_image_paths, find_class,
    show_text_segmentation, show_visual_segmentation,
    show_instance_grid, show_score_sweep,
    show_gallery_reference, show_multiclass_segmentation,
)
print("Imports OK.")

In [ ]:
# ── Edit these two paths ───────────────────────────────────────────────────────
DATASET_PATH = pathlib.Path(r"../fruits-360-100x100")
SCENE_PATH   = pathlib.Path(r"apple_pear_1.jpg")
# ──────────────────────────────────────────────────────────────────────────────

TRAIN_ROOT = DATASET_PATH / "Training"
assert TRAIN_ROOT.exists(), f"Training/ not found under {DATASET_PATH}"
assert SCENE_PATH.exists(), f"Scene image not found: {SCENE_PATH}"

classes = sorted([d.name for d in TRAIN_ROOT.iterdir() if d.is_dir()])

scene_img      = Image.open(SCENE_PATH).convert("RGB")
orig_w, orig_h = scene_img.size
scene_arr      = np.array(scene_img)

print(f"Dataset: {len(classes)} classes")
print(f"Scene  : {SCENE_PATH.name}  {orig_w} × {orig_h} px")

## Authenticate with Hugging Face

SAM3 weights require a free Hugging Face account and an accepted model license.

1. Create an account at huggingface.co
2. Go to **Settings → Access Tokens → New token** (read permission is enough)
3. Accept the model license on the [facebook/sam3](https://huggingface.co/facebook/sam3) page
4. Paste your token in the cell below

In [ ]:
from huggingface_hub import login
token = "my_private_token_which_I_won't_share!!!!!!"
login(token=token, add_to_git_credential=False)

## Load Models

SAM3 ships two distinct model heads:

| Class | Prompt types | Post-process call |
|---|---|---|
| `Sam3Model` | text, text + box/point | `post_process_instance_segmentation` |
| `Sam3TrackerModel` | point, box | `post_process_masks` |

We load both here so every mode in the notebook is ready to run.

In [ ]:
print(torch.__version__)  
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

sam3, proc3       = load_sam3_model(MODEL_ID, DEVICE)     # text-prompted
tracker, proc_trk = load_tracker_model(MODEL_ID, DEVICE)  # visual-prompted

---
## Part 1 — Text-Prompted Segmentation

SAM3's open-vocabulary text encoder matches the prompt against all patches in the
image and produces a mask for every detected instance. No reference images, no manual
boxes — just a class name string.

```
text prompt (e.g. "apple")
        │
        ▼  text encoder + mask decoder
N instance masks  +  IoU confidence scores
        │
        ▼  score threshold filter
final detections
```

We run inference at a **low threshold (0.1)** to collect all candidate masks, then
explore different filtering thresholds in Part 1b without re-running the model.

### 1a — Single Class

In [ ]:
TEXT_PROMPT    = "apple"    # change to any fruit name present in the scene
SCORE_THRESHOLD = 0.50      # keep instances above this IoU confidence

In [ ]:
all_dets, _ = segment_by_text(
    SCENE_PATH, sam3, proc3, TEXT_PROMPT, DEVICE, threshold=0.10
)
dets = [d for d in all_dets if d["score"] >= SCORE_THRESHOLD]

print(f"Total candidates (threshold=0.10) : {len(all_dets)}")
print(f"After filtering  (>= {SCORE_THRESHOLD})    : {len(dets)}")
for i, d in enumerate(dets):
    print(f"  #{i+1:2d}  score={d['score']:.4f}  "
          f"mask pixels={d['mask'].sum():,}  box={d['box'].astype(int).tolist()}")

In [ ]:
show_text_segmentation(
    scene_arr, dets, TEXT_PROMPT,
    save_path="sam3_text_single.png",
)

In [ ]:
show_instance_grid(
    scene_arr, dets, n_cols=4,
    save_path="sam3_instance_grid.png",
)

### 1b — Open-Vocabulary Multi-Class Segmentation

Run a separate text query for each fruit type present in the scene. Because SAM3 is
trained on 270K+ concepts, it can distinguish "apple" from "pear" without any
task-specific fine-tuning.

In [ ]:
# Adjust to the fruit classes visible in your scene
SCENE_CLASSES   = # TODO
MULTICLASS_THR  = 0.50

In [ ]:
detections_per_class = {}
for cls in SCENE_CLASSES:
    all_d, _ = segment_by_text(
        SCENE_PATH, sam3, proc3, cls, DEVICE, threshold=0.10
    )
    detections_per_class[cls] = [d for d in all_d if d["score"] >= MULTICLASS_THR]
    print(f"'{cls}': {len(detections_per_class[cls])} instance(s) after threshold")

show_multiclass_segmentation(
    scene_arr, detections_per_class,
    save_path="sam3_multiclass.png",
)

### 1c — Gallery-Informed Text Prompt

The Fruits360 class names are already valid text concepts for SAM3. We can view the
reference images from the training split (as we did for the DINOv3 gallery prototype)
simply as context — then use the class name directly as the text prompt.

No feature extraction or prototype averaging is needed.

In [ ]:
GALLERY_CLASS = "Apple Red"   # prefix match — change to any Fruits360 class
N_REF         = 6
GALLERY_THR   = 0.50

In [ ]:
ref_class = find_class(GALLERY_CLASS, classes)
ref_paths = get_image_paths(TRAIN_ROOT, ref_class, N_REF)
print(f"Using class: {ref_class}")

show_gallery_reference(ref_paths, ref_class, save_path="sam3_gallery_refs.png")

# Use the base concept word (first token of the class name) as the text prompt
gallery_prompt = ref_class.split()[0].lower()   # e.g. "Apple Red" → "apple"
print(f"Text prompt derived from class: '{gallery_prompt}'")

gallery_dets, _ = segment_by_text(
    SCENE_PATH, sam3, proc3, gallery_prompt, DEVICE, threshold=0.10
)
gallery_dets = [d for d in gallery_dets if d["score"] >= GALLERY_THR]

show_text_segmentation(
    scene_arr, gallery_dets, gallery_prompt,
    save_path="sam3_gallery_text.png",
)

---
## Part 2 — Bounding Box Segmentation

Draw a rough bounding box around **one** fruit in the scene. `Sam3TrackerModel`
predicts a precise pixel-level mask for the object inside the box.

Advantages over the text mode:
- Works when the text concept is ambiguous or not in SAM3's vocabulary
- Targets a *specific* instance rather than all instances of a class
- Typically yields higher IoU for single-object queries

In [ ]:
# [x1, y1, x2, y2] in original image pixels — draw a box around one fruit
QUERY_BOX = [1938, 1054, 2538, 1800]

In [ ]:
x1, y1, x2, y2 = QUERY_BOX
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(scene_arr)
ax.add_patch(mpatches.Rectangle(
    (x1, y1), x2 - x1, y2 - y1,
    linewidth=2.5, edgecolor="#e74c3c", facecolor="none", linestyle="--"))
ax.set_title(f"Query box  [{x1}, {y1}, {x2}, {y2}]", fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
bbox_result, _ = segment_by_bbox(SCENE_PATH, tracker, proc_trk, QUERY_BOX, DEVICE)
print(f"Mask pixels : {bbox_result['mask'].sum():,} / {orig_w * orig_h:,}")
print(f"IoU score   : {bbox_result['score']:.4f}")

show_visual_segmentation(
    scene_arr, bbox_result, "bbox",
    query_box=QUERY_BOX,
    save_path="sam3_bbox.png",
)

---
## Part 3 — Point-Click Segmentation

Click a single pixel inside the target object. `Sam3TrackerModel` predicts the mask
for the object whose surface contains that pixel.

This is the fastest prompt mode but also the least robust: if the clicked pixel lands
on a shadow, specular highlight, or object boundary, the mask quality drops.
Use `multimask_output=True` (not shown here) to get three candidate masks and pick
the one with the highest IoU score.

In [ ]:
# (x, y) in original image pixels — click inside the target fruit
QUERY_POINT = (2150, 1400)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(scene_arr)
ax.plot(QUERY_POINT[0], QUERY_POINT[1], "r+", markersize=14, markeredgewidth=2.5)
ax.set_title(f"Query point  {QUERY_POINT}", fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
point_result, _ = segment_by_point(SCENE_PATH, tracker, proc_trk, QUERY_POINT, DEVICE)
print(f"Mask pixels : {point_result['mask'].sum():,} / {orig_w * orig_h:,}")
print(f"IoU score   : {point_result['score']:.4f}")

show_visual_segmentation(
    scene_arr, point_result, "point",
    query_point=QUERY_POINT,
    save_path="sam3_point.png",
)

### 3a — Positive + Negative Point Refinement

Pass multiple points: positive (label=1) to include, negative (label=0) to exclude.
This manually steers the mask away from background or adjacent objects.

In [ ]:
# Add a negative point outside the target fruit to prevent mask bleed
POSITIVE_POINT  = (2150, 1400)   # inside the target
NEGATIVE_POINT  = (2600, 1400)   # background or adjacent fruit

In [ ]:
from sam3_seg_utils import _tracker_score

refined_img = Image.open(SCENE_PATH).convert("RGB")
refined_inputs = None # TODO

with torch.no_grad():
    refined_outputs = tracker(**refined_inputs, multimask_output=False)

refined_masks = proc_trk.post_process_masks(
    refined_outputs.pred_masks.cpu(), refined_inputs["original_sizes"]
)[0]
refined_result = {
    "mask":  refined_masks[0, 0].numpy().astype(bool),
    "score": _tracker_score(refined_outputs),
}

print(f"IoU score (refined) : {refined_result['score']:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(scene_arr)
ax.plot(*POSITIVE_POINT, "g+", markersize=14, markeredgewidth=2.5, label="positive")
ax.plot(*NEGATIVE_POINT, "r+", markersize=14, markeredgewidth=2.5, label="negative")
ax.legend(fontsize=9)
ax.set_title("Positive + negative point prompts", fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.show()

show_visual_segmentation(
    scene_arr, refined_result, "point (refined)",
    query_point=POSITIVE_POINT,
    save_path="sam3_point_refined.png",
)

---
## Part 4 — SAM3 vs DINOv3: Comparison

Both approaches use the same scene and target concept. The side-by-side comparison
makes the precision gap visible.

| Aspect | DINOv3 cosine threshold | SAM3 |
|---|---|---|
| **Boundary precision** | ±16 px (one patch) | Sub-pixel (learned upsampling) |
| **Prompt type** | Reference image patches | Text / point / box |
| **Multi-instance** | One heatmap for all similar patches | Distinct mask per instance |
| **Threshold** | Manual percentile — scene-dependent | Learned IoU score — interpretable |
| **Background bleed** | Leaks to similarly-coloured regions | Object-shape prior in decoder |
| **Confidence** | Raw cosine similarity | Calibrated IoU predictions |
| **Inference speed** | Fast (one forward pass) | Moderate (mask decoder per object) |

In [ ]:
dino_mask_path = pathlib.Path("detection_gallery.png")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

axes[0].imshow(scene_arr)
axes[0].set_title("Original scene", fontsize=12)
axes[0].axis("off")

if dino_mask_path.exists():
    axes[1].imshow(Image.open(dino_mask_path))
    axes[1].set_title(
        "DINOv3 cosine threshold\n(single heatmap, coarse boundaries)", fontsize=11)
else:
    axes[1].text(0.5, 0.5, "Run DINOv3 notebook first\n(detection_gallery.png)",
                 ha="center", va="center", transform=axes[1].transAxes, fontsize=10)
    axes[1].set_title("DINOv3 (not available)", fontsize=11)
axes[1].axis("off")

axes[2].imshow(_color_overlay(scene_arr, dets))
for i, det in enumerate(dets):
    axes[2].contour(det["mask"].astype(float), levels=[0.5],
                    colors=[INSTANCE_COLORS[i % len(INSTANCE_COLORS)]], linewidths=0.9)
axes[2].set_title(
    f"SAM3 text '{TEXT_PROMPT}'\n({len(dets)} instances, precise boundaries)", fontsize=11)
axes[2].axis("off")

plt.suptitle("DINOv3 patch similarity  vs  SAM3 instance segmentation",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("sam3_vs_dinov3.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Summary

### What SAM3 gives you that DINOv3 threshold segmentation cannot

1. **Instance separation** — each detected fruit gets its own binary mask; touching
   apples are separate objects, not one blob.
2. **Pixel-accurate boundaries** — the learned mask decoder recovers fine contours that
   the 14×14 DINOv3 patch grid fundamentally cannot represent.
3. **Calibrated confidence** — the IoU prediction score tells you *how certain the model
   is* about each mask, making automated filtering straightforward.
4. **No threshold tuning** — the text or visual prompt replaces the manual
   `SIM_PERCENTILE` knob; the score threshold has a consistent semantic meaning across
   scenes and classes.
5. **Open-vocabulary** — any English noun phrase works as a prompt; no retraining
   or adapter is required for new fruit classes.

### When DINOv3 patch similarity is still useful

- Extremely fast inference with no learned decoder overhead  
- Works with patch-level similarity for dense cross-image retrieval and patch matching  
- Produces a graded heatmap rather than a hard mask, useful for attention visualisation  
- Lower GPU memory footprint at batch scale